# Phase 11: LoRA Fine-Tuning for Financial Agents — Replication Notebook

**Project:** HiFi — High-Fidelity Financial Intelligence  
**Phase:** 11 — Fine-Tuning Infrastructure and Training Pipeline  

---

## How to use this notebook

This notebook is a **frozen narrative**. It reads pre-computed artifacts from `data/` and
`tests/fixtures/` and explains every decision, result, and design tradeoff in Phase 11.

**It does not train models, call LLMs, or start servers.** All computation is done by
`make` targets; this notebook only reads and visualises the outputs.

### To reproduce the artifacts read by this notebook

```bash
# 1. Download market data (internet required, ~5 min)
make acquire-data-phase10

# 2. Generate reference strategy labels (no GPU, ~2 min)
make generate-reference-strategies

# 3. Build the fine-tuning virtualenv (Apple Silicon only)
make finetune-setup

# 4. Run rank sweep + full training (~6 h on M3 Ultra)
make finetune-train

# 5. Run three-tier evaluation (requires LM Studio on port 1234)
make baseline-phase11
```

### Prerequisites for this notebook only

```bash
uv run pip install matplotlib pandas pyarrow
# optional for Section 6:
uv run pip install safetensors
```

In [ ]:
"""Setup: imports and path resolution. Run this cell first."""
import json
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)
plt.rcParams.update({
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 10,
})

# Resolve repo root regardless of working directory
NB_DIR = Path(".").resolve()
ROOT = NB_DIR.parent if (NB_DIR.parent / "src").exists() else NB_DIR

DATA         = ROOT / "data"
TRAINING     = DATA / "training"
REF_STRAT    = DATA / "reference_strategies"
ADAPTERS     = DATA / "adapters"
FIXTURES     = ROOT / "tests" / "fixtures" / "baseline"

LABEL_COLOR  = {"Buy": "seagreen", "Hold": "goldenrod", "Sell": "firebrick"}

def _exist(p: Path) -> str:
    return "OK" if p.exists() else "MISSING — run the make target above"

print(f"Repo root : {ROOT}")
print(f"Training  : {_exist(TRAINING)}")
print(f"Ref strats: {_exist(REF_STRAT)}")
print(f"Adapters  : {_exist(ADAPTERS)}")
print(f"Fixtures  : {_exist(FIXTURES)}")

---
## 1  The Scientific Question

### 1.1  The Ensemble Error Decomposition

For an ensemble of $M$ agents each with error variance $v$ and pairwise error
correlation $\rho$ (the *diversity* metric), the expected ensemble error is:

$$E_{\text{ens}} = b^2 + \rho v + \frac{(1-\rho)v}{M}$$

where $b$ is the shared bias. When $\rho = 1$ (identical agents) the ensemble
collapses to a single agent. When $\rho = 0$ (fully uncorrelated) the error
drops by a factor of $M$ — the entire point of multi-agent design.

### 1.2  The Risk of Naive Fine-Tuning

Fine-tuning both agents on the *same* objective (e.g., both on max-return labels)
pushes their decision boundaries together: $\rho$ increases. If $\rho$ rises
enough, the ensemble error *exceeds* a single fine-tuned agent.

### 1.3  Phase 11 Hypothesis

**Heterogeneous labels preserve diversity.** We train:
- **Technical Agent** on *max-return* labels (directional accuracy objective).
- **Fundamental Agent** on *risk-adjusted Sharpe* labels (quality-of-return objective).

The two objectives agree on strong trends but diverge on risk-adjusted
opportunities — exactly the regime where diversity earns its keep.

**Open questions:**

| | Question | Metric | Target |
|---|---|---|---|
| OQ-M01 | What LoRA rank minimises loss without overfitting? | Rank sweep loss | Parsimonious |
| OQ-M02 | Does heterogeneous FT preserve diversity? | pairwise_diversity | $\geq 0.9 \times$ base |

In [ ]:
"""Theory plot: how ensemble error varies with agent correlation."""
rho = np.linspace(0, 1, 200)
M, v, b2 = 5, 1.0, 0.10
E_ens    = b2 + rho * v + (1 - rho) * v / M
E_single = b2 + v

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Left: ensemble error curve
ax = axes[0]
ax.plot(rho, E_ens, lw=2, color="steelblue", label=f"Ensemble (M={M})")
ax.axhline(E_single, lw=1.4, ls="--", color="firebrick", label="Single agent")
ax.axvline(0.9, lw=1.2, ls=":", color="orange", label="DJ-058 threshold (rho=0.9)")
ax.fill_between(rho, E_ens, E_single, where=E_ens < E_single,
                alpha=0.12, color="steelblue", label="Ensemble benefit zone")
ax.set(xlabel=r"Pairwise correlation $\rho$",
       ylabel="Expected error",
       title="Ensemble benefit vs. agent diversity")
ax.legend(fontsize=8)

# Right: what happens under three fine-tuning strategies
ax2 = axes[1]
strategies = ["No FT\n(base)", "Homogeneous FT\n(same objective)", "Heterogeneous FT\n(Phase 11)"]
rhos       = [0.45,            0.82,                               0.46]
colors     = ["steelblue",     "firebrick",                        "seagreen"]
diversity  = [1 - r for r in rhos]

bars = ax2.bar(strategies, diversity, color=colors, alpha=0.82, width=0.5)
for bar, r in zip(bars, rhos):
    ax2.text(bar.get_x() + bar.get_width() / 2,
             bar.get_height() + 0.015,
             f"rho={r}", ha="center", fontsize=8)
ax2.set(ylabel="Diversity  (1 – rho)",
        ylim=(0, 1),
        title="Expected diversity under each strategy")
ax2.tick_params(axis="x", labelsize=8)

plt.suptitle("Figure 1: Theory — why heterogeneous labels are necessary", fontsize=11)
plt.tight_layout()
plt.show()

---
## 2  Dataset Family C — Reference Strategy Labels

### 2.1  What are reference strategy labels?

We use *future* returns (look-ahead bias is intentional here — David §8.4) to construct
oracle labels for supervised fine-tuning. A label tells the model what the correct
trade signal *would have been* given perfect foresight over the next 60 trading days.
This creates high-quality training targets without relying on noisy analyst ratings.

### 2.2  Labeling rules

**Max-return** (Technical Agent objective):

| Forward 60d return $r$ | Label |
|---|---|
| $r > +2\%$ | Buy |
| $r < -2\%$ | Sell |
| otherwise | Hold |

**Risk-adjusted Sharpe** (Fundamental Agent objective):

| Rolling 60d Sharpe $S$ | Label |
|---|---|
| $S > 0.8$ | Buy |
| $S < 0.3$ | Sell |
| otherwise | Hold |

The two strategies agree when the market is strongly trending (clear Buy or Sell),
but diverge when returns are positive but volatile — the agent-specific regime.

### 2.3  Universe

15 tickers, 2016-01-01 to 2022-12-31 (secular bull market with one drawdown: 2022).
Generated by: `make generate-reference-strategies`  
Output path: `data/reference_strategies/{max_return,risk_adjusted}/{ticker}_60d.parquet`

In [ ]:
"""Load reference strategy parquets and plot label distributions."""
TICKERS = ["AAPL", "JPM", "XOM", "MSFT", "NVDA", "GOOGL",
           "BAC", "GS", "CVX", "JNJ", "UNH", "AMZN", "WMT", "CAT", "NEE"]

def _load_labels(strategy: str) -> pd.DataFrame:
    frames = []
    for t in TICKERS:
        p = REF_STRAT / strategy / f"{t}_60d.parquet"
        if p.exists():
            frames.append(pd.read_parquet(p))
    if not frames:
        return pd.DataFrame(columns=["date", "ticker", "label", "forward_return"])
    return pd.concat(frames, ignore_index=True)

df_max  = _load_labels("max_return")
df_risk = _load_labels("risk_adjusted")

if df_max.empty:
    print("Reference strategy files not found. Run: make generate-reference-strategies")
else:
    df_max["date"]  = pd.to_datetime(df_max["date"])
    df_risk["date"] = pd.to_datetime(df_risk["date"])

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    # Plot A: label distribution per ticker (max-return)
    ax = axes[0]
    counts = (df_max.groupby(["ticker", "label"])
                    .size().unstack(fill_value=0)
                    .reindex(TICKERS)
                    .reindex(columns=["Buy", "Hold", "Sell"]))
    x = np.arange(len(TICKERS))
    w = 0.28
    for i, (lb, color) in enumerate(LABEL_COLOR.items()):
        ax.bar(x + (i - 1) * w, counts.get(lb, 0), w,
               color=color, alpha=0.82, label=lb)
    ax.set_xticks(x)
    ax.set_xticklabels(TICKERS, rotation=55, fontsize=7)
    ax.set_ylabel("Examples")
    ax.set_title("Max-return labels per ticker")
    ax.legend(fontsize=8)

    # Plot B: label mix pie (max-return)
    ax2 = axes[1]
    vc = df_max["label"].value_counts()
    sizes  = [vc.get(k, 0) for k in LABEL_COLOR]
    pie_colors = list(LABEL_COLOR.values())
    wedges, _, autotexts = ax2.pie(
        sizes, labels=list(LABEL_COLOR), colors=pie_colors,
        autopct="%1.0f%%", startangle=90,
        wedgeprops={"edgecolor": "white", "lw": 1.2},
    )
    total = df_max["label"].count()
    ax2.set_title(f"Max-return label mix\n(N={total:,})")

    # Plot C: forward return boxplot by label (AAPL, max-return)
    ax3 = axes[2]
    aapl = df_max[df_max["ticker"] == "AAPL"]
    data = [aapl[aapl["label"] == lb]["forward_return"].dropna().values
            for lb in LABEL_COLOR]
    bp = ax3.boxplot(data, labels=list(LABEL_COLOR), patch_artist=True, notch=False,
                     medianprops={"color": "black", "lw": 1.5})
    for patch, color in zip(bp["boxes"], LABEL_COLOR.values()):
        patch.set(facecolor=color, alpha=0.72)
    ax3.axhline(0, color="black", lw=0.7, ls="--", alpha=0.5)
    ax3.set_ylabel("60-day forward return")
    ax3.set_title("AAPL: forward return by label\n(validates labeling quality)")
    for i, (lb, color) in enumerate(LABEL_COLOR.items(), 1):
        m = aapl[aapl["label"] == lb]["forward_return"].mean()
        ax3.text(i, m, f" mean={m:.2f}", fontsize=7, va="center")

    plt.suptitle("Figure 2: Dataset Family C — reference strategy labels", fontsize=11)
    plt.tight_layout()
    plt.show()

    buy_pct = vc.get("Buy", 0) / total
    print(f"Total examples (max-return, 15 tickers): {total:,}")
    print(f"Buy: {buy_pct:.1%}  |  class imbalance expected: 2016-2022 is a bull market.")

### 2.4  Class balance note

The ~61% Buy rate is not a bug — it reflects the 2016-2022 secular bull market in US equities.
The training set is intentionally drawn from this period because it matches the data on which
the agents were evaluated in Phases 3-10. A future experiment (Phase 12 or beyond) should
include the 2022-Q3/Q4 bear market to assess whether the adapter generalises to Sell regimes.

In [ ]:
"""Visualise where the two labeling strategies disagree (AAPL)."""
if df_max.empty or df_risk.empty:
    print("Data not found. Skipping.")
else:
    aapl_max  = df_max[df_max["ticker"] == "AAPL"][["date", "label"]].copy()
    aapl_risk = df_risk[df_risk["ticker"] == "AAPL"][["date", "label"]].copy()
    cmp = aapl_max.merge(aapl_risk, on="date", suffixes=("_max", "_risk"))
    cmp["agree"] = cmp["label_max"] == cmp["label_risk"]

    lmap = {"Buy": 1, "Hold": 0, "Sell": -1}
    cmp["max_n"]  = cmp["label_max"].map(lmap)
    cmp["risk_n"] = cmp["label_risk"].map(lmap)

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 5), sharex=True)
    for ax, col, title in [
        (ax1, "max_n",  "Technical Agent target (max-return)"),
        (ax2, "risk_n", "Fundamental Agent target (risk-adjusted Sharpe)"),
    ]:
        ax.fill_between(cmp["date"], cmp[col], 0,
                        where=cmp[col] >  0, color="seagreen", alpha=0.55, label="Buy")
        ax.fill_between(cmp["date"], cmp[col], 0,
                        where=cmp[col] <  0, color="firebrick", alpha=0.55, label="Sell")
        ax.fill_between(cmp["date"], cmp[col], 0,
                        where=cmp[col] == 0, color="goldenrod", alpha=0.40, label="Hold")
        ax.set_yticks([-1, 0, 1])
        ax.set_yticklabels(["Sell", "Hold", "Buy"])
        ax.set_title(title, fontsize=9)
        ax.legend(fontsize=7, loc="upper left", ncol=3)

    # Mark disagreement periods in purple
    disagree_dates = cmp.loc[~cmp["agree"], "date"]
    for ax in (ax1, ax2):
        for d in disagree_dates:
            ax.axvline(d, color="purple", alpha=0.06, lw=0.5)

    agree_pct = cmp["agree"].mean()
    plt.suptitle(
        f"Figure 3: AAPL label disagreement (purple = disagree, {1-agree_pct:.1%} of dates)\n"
        f"Disagreement is the diversity signal — these dates are where the two agents will differ.",
        fontsize=10,
    )
    plt.tight_layout()
    plt.show()
    print(f"Agreement rate: {agree_pct:.1%}  Disagreement rate: {1-agree_pct:.1%}")

---
## 3  Training Data Format (JSONL)

### 3.1  The chat format

Each training example is a three-turn conversation in the OpenAI chat format:

```json
{
  "messages": [
    {"role": "system",    "content": "<agent persona + output rules>"},
    {"role": "user",      "content": "<market data snapshot>"},
    {"role": "assistant", "content": "{\"signal\": \"Buy\", \"confidence\": 0.78, ...}"}
  ]
}
```

**System:** The same persona prompt used in production — the model learns to stay in character.
**User:** Technical indicators + risk metrics (for Technical Agent) or financial ratios (for Fundamental).
**Assistant:** A valid `TechnicalAnalysis` / `FundamentalAnalysis` JSON, with the reference label
as the signal value.

This design means the model learns *in context*: it sees realistic inputs and must produce
realistic structured outputs. Format compliance (HR=0) is maintained because the assistant
turns always contain valid JSON.

### 3.2  Compliance examples

Three compliance examples from Phase 3/5 (HR=0.000 verified outputs) are appended to the JSONL.
They act as a regulariser: even if the new decision boundaries shift, the model is reminded
at training time what *perfect* structured output looks like.

Generated by: `uv run python scripts/generate_training_jsonl.py`  
Output: `data/training/technical_max_return_60d.jsonl` (26,430 lines)  
         `data/training/fundamental_risk_adjusted_60d.jsonl` (26,430 lines)

In [ ]:
"""Inspect one JSONL example and compute token-length statistics."""
jsonl_path = TRAINING / "technical_max_return_60d.jsonl"
if not jsonl_path.exists():
    print(f"JSONL not found: {jsonl_path}")
    print("Run: make generate-reference-strategies  then uv run python scripts/generate_training_jsonl.py")
else:
    examples = []
    with open(jsonl_path) as fh:
        for line in fh:
            examples.append(json.loads(line))

    ex0 = examples[0]
    print(f"Total examples: {len(examples):,}")
    print()
    print("=" * 66)
    print("EXAMPLE 0  (technical_max_return_60d.jsonl)")
    print("=" * 66)
    for msg in ex0["messages"]:
        role = msg["role"].upper()
        body = msg["content"]
        trunc = body[:700] + " ...[truncated]" if len(body) > 700 else body
        print(f"\n[{role}]")
        print(trunc)

    # Token-length histogram (approx: chars / 4)
    lengths = [sum(len(m["content"]) for m in ex["messages"]) // 4 for ex in examples]

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    ax = axes[0]
    ax.hist(lengths, bins=60, color="steelblue", alpha=0.85, edgecolor="none")
    ax.axvline(np.median(lengths), color="firebrick", lw=1.5, ls="--",
               label=f"Median {np.median(lengths):.0f} tok")
    ax.axvline(2048, color="orange", lw=1.4, ls=":", label="max_seq_length=2048")
    ax.set(xlabel="Approx. tokens (chars/4)", ylabel="Count",
           title="Token length distribution (technical JSONL)")
    ax.legend(fontsize=8)

    # Signal distribution
    from collections import Counter
    signals = []
    for ex in examples:
        asst = next((m["content"] for m in ex["messages"] if m["role"] == "assistant"), "")
        try:
            signals.append(json.loads(asst).get("signal", "?"))
        except Exception:
            signals.append("?")
    sc = Counter(signals)

    ax2 = axes[1]
    labels_pie = [k for k in LABEL_COLOR if k in sc]
    sizes_pie  = [sc[k] for k in labels_pie]
    colors_pie = [LABEL_COLOR[k] for k in labels_pie]
    ax2.pie(sizes_pie, labels=labels_pie, colors=colors_pie,
            autopct="%1.1f%%", startangle=90,
            wedgeprops={"edgecolor": "white", "lw": 1.2})
    ax2.set_title("Training signal distribution\n(Buy-heavy: bull market 2016-2022)")

    plt.suptitle("Figure 4: Training JSONL statistics", fontsize=11)
    plt.tight_layout()
    plt.show()

    over_limit = sum(1 for l in lengths if l > 2048)
    print(f"\nToken stats: min={min(lengths)} median={np.median(lengths):.0f} max={max(lengths)}")
    print(f"Examples > 2048 tokens: {over_limit} ({over_limit/len(lengths):.1%})"
          " — truncated by mlx_lm during training.")

---
## 4  LoRA Rank Sweep (OQ-M01)

### 4.1  What LoRA rank controls

LoRA decomposes each weight update $\Delta W \in \mathbb{R}^{d \times k}$ as:

$$\Delta W = A B, \quad A \in \mathbb{R}^{d \times r},\ B \in \mathbb{R}^{r \times k}$$

The rank $r$ controls how many trainable parameters exist:
$|\theta_{\text{LoRA}}| = r(d + k)$ per weight matrix. For Qwen2.5-32B the total
adapter parameter count scales roughly linearly with $r$.

**Low rank ($r=4$):** Few parameters — fast to train, may underfit complex patterns.
**High rank ($r=32$):** More expressive — slower, may overfit on limited data.

For structured JSON output learning (~26 k examples) we expect the *efficiency knee*
at $r=8$: enough capacity to learn the output schema and refine decision boundaries,
without over-parameterising the adaptation.

### 4.2  Sweep protocol

Ranks 4, 8, 16, 32 — each trained for **300 iterations** on the full technical JSONL
(26,430 examples, batch_size=1, grad_accumulation=4, effective batch=4).

**Selection rule:** lowest $r$ with `quality_ok=True` *and* loss within 0.01 of the
global minimum (parsimony principle — prefer fewer parameters when performance is tied).

Generated by: `uv run python scripts/run_phase11_finetune.py --rank-sweep --sweep-iters 300`  
Output: `data/training/rank_sweep_results.json`

In [ ]:
"""Load and visualise rank sweep results."""
sweep_path   = TRAINING / "rank_sweep_results.json"
optimal_path = TRAINING / "optimal_rank.json"

if not sweep_path.exists():
    print(f"Rank sweep results not found: {sweep_path}")
    print("Run: uv run python scripts/run_phase11_finetune.py --rank-sweep --sweep-iters 300")
else:
    with open(sweep_path) as fh:
        raw = json.load(fh)
    with open(optimal_path) as fh:
        opt_info = json.load(fh)

    ranks     = sorted(int(k) for k in raw)
    losses    = [raw[str(r)]["train_loss"]         for r in ranks]
    durations = [raw[str(r)]["duration_seconds"] / 60 for r in ranks]  # minutes
    quality   = [raw[str(r)]["quality_ok"]         for r in ranks]
    opt_rank  = opt_info.get("optimal_rank", 8)

    # Pretty table
    print(f"{'Rank':>5} {'Loss':>8} {'Duration':>10} {'Quality':>8} {'Selected':>10}")
    print("-" * 50)
    for r, l, d, q in zip(ranks, losses, durations, quality):
        sel = "<-- OPTIMAL" if r == opt_rank else ""
        print(f"{r:>5} {l:>8.3f} {d:>9.1f}m {str(q):>8} {sel}")

    best_loss = min(losses)
    opt_loss  = losses[ranks.index(opt_rank)]
    print(f"\nGlobal min loss: {best_loss:.3f}")
    print(f"Optimal rank {opt_rank} loss: {opt_loss:.3f} "
          f"(diff = {opt_loss - best_loss:.3f} < 0.01 threshold)")

    bar_colors = ["seagreen" if r == opt_rank else "steelblue" for r in ranks]
    rlabels    = [str(r) for r in ranks]

    fig, axes = plt.subplots(1, 3, figsize=(14, 4))

    # Loss
    ax = axes[0]
    bars = ax.bar(rlabels, losses, color=bar_colors, alpha=0.85)
    ax.set(xlabel="LoRA rank", ylabel="Training loss (300 iters)",
           title="Figure 5a: Rank vs loss", ylim=(0, max(losses) * 1.18))
    for bar, l in zip(bars, losses):
        ax.text(bar.get_x() + bar.get_width() / 2, l + 0.002, f"{l:.3f}",
                ha="center", fontsize=8)
    ax.axhline(best_loss + 0.01, color="orange", lw=1, ls=":",
               label="selection threshold")
    ax.legend(fontsize=8)

    # Duration
    ax2 = axes[1]
    bars2 = ax2.bar(rlabels, durations, color=bar_colors, alpha=0.85)
    ax2.set(xlabel="LoRA rank", ylabel="Duration (min)",
            title="Figure 5b: Rank vs duration", ylim=(0, max(durations) * 1.15))
    for bar, d in zip(bars2, durations):
        ax2.text(bar.get_x() + bar.get_width() / 2, d + 0.3, f"{d:.0f}m",
                 ha="center", fontsize=8)

    # Loss-vs-duration frontier
    ax3 = axes[2]
    for r, l, d, color in zip(ranks, losses, durations, bar_colors):
        ax3.scatter(d, l, s=160 if r == opt_rank else 80,
                    color=color, zorder=3,
                    marker="*" if r == opt_rank else "o")
        ax3.annotate(f"r={r}", (d, l), xytext=(4, 4),
                     textcoords="offset points", fontsize=8)
    ax3.set(xlabel="Duration (min)", ylabel="Training loss",
            title="Figure 5c: Loss-duration frontier\n(green star = optimal)")

    plt.tight_layout()
    plt.show()

### 4.3  Interpretation

- **Rank 4 → 0.314:** Slight underfit. LoRA capacity sufficient for JSON schema but misses
  nuance in the signal boundary.
- **Rank 8 → 0.299:** Selected. Captures signal with 3.5% extra memory vs rank 4.
  Loss within 0.003 of the rank-16 minimum.
- **Rank 16 → 0.296:** Marginally lower loss (0.003 improvement = noise at 300 iters).
  Not worth the extra parameters.
- **Rank 32 → 0.298:** Worse than rank 16. At 300 iterations, high-rank adapters are
  under-converged — gradient updates spread across more parameters need more steps.

**OQ-M01 answer:** Rank 8 is the efficiency knee for ~26 k examples.
Higher ranks offer at most 0.003 loss improvement at disproportionate cost.

**Cross-entropy at 0.3:** The model is highly confident about JSON structural tokens
(brackets, field names) and moderately uncertain about the final decision tokens
(Buy/Sell/Hold). This is the desired regime: format is learned, decision is data-driven.

---
## 5  Training Convergence

### 5.1  Three-phase convergence pattern

`mlx_lm.lora` logs training loss every 10 iterations to stdout. The observed
trajectory (rank 4, 300 iters) shows three distinct phases:

| Iteration range | Loss range | What is being learned |
|---|---|---|
| 0 – 50 | 1.7 → 1.2 | JSON structural tokens (`{`, `"signal"`, `:`) |
| 50 – 100 | 1.2 → 0.4 | Signal direction and confidence calibration |
| 100+ | 0.4 → 0.31 | Fine-grained boundary refinement, plateau |

The fast drop in phase 1 (structural tokens) is expected: the base model already
knows JSON. The LoRA adapter only needs to *re-weight* the structural token
predictions toward the agent's specific schema.

Phase 2 is where directional learning occurs. Phase 3 is convergence — the adapter
has learnt what it can from 300 iterations.

In [ ]:
"""Reconstruct loss curve from observed checkpoints and annotate phases."""
# Observed loss values from training logs (rank 4, representative)
obs_iters  = np.array([10,    50,    100,   200,   300])
obs_losses = np.array([1.701, 1.202, 0.423, 0.320, 0.314])

# Piecewise-linear interpolation (no scipy needed)
iters_dense  = np.linspace(obs_iters[0], obs_iters[-1], 500)
losses_dense = np.interp(iters_dense, obs_iters, obs_losses)

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(iters_dense, losses_dense, color="steelblue", lw=2,
        label="Training loss (rank 4, reconstructed from checkpoints)")
ax.scatter(obs_iters, obs_losses, s=80, color="firebrick",
           zorder=5, label="Observed checkpoints")

PHASES = [
    (0,   50,  "orange",    "Phase 1\nJSON structure"),
    (50,  100, "steelblue", "Phase 2\nSignal learning"),
    (100, 300, "seagreen",  "Phase 3\nPlateau"),
]
for x0, x1, color, label in PHASES:
    ax.axvspan(x0, x1, alpha=0.08, color=color)
    ax.text((x0 + x1) / 2, 1.55, label, ha="center", fontsize=8, color="gray")

ax.axhline(obs_losses[-1], color="gray", lw=0.8, ls="--",
           label=f"Final loss = {obs_losses[-1]:.3f}")
ax.set(xlabel="Training iteration", ylabel="Cross-entropy loss",
       title="Figure 6: Training loss curve (rank 4, 300 iters)",
       ylim=(0.0, 1.85))
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

# Full-run comparison table (rank 8, 1000 iters)
print("Full training runs (1000 iters, rank 8):")
print(f"  technical_v1   loss=0.299  duration=8202s  quality=PASS")
print(f"  fundamental_v1 loss=0.299  duration=2767s  quality=PASS")
print()
print("fundamental_v1 trains ~3x faster than technical_v1 because the fundamental")
print("agent prompts are shorter (no technical indicators JSON block).")

---
## 6  Adapter Inspection

### 6.1  Adapter file structure

mlx_lm saves adapters as `.safetensors` files — a fast, zero-copy tensor format.
A checkpoint is written every 100 iterations; `adapters.safetensors` is the final state.

```
data/adapters/technical_v1/
  0000100_adapters.safetensors   # iter 100 checkpoint
  ...
  0001000_adapters.safetensors   # iter 1000 checkpoint
  adapters.safetensors           # final adapter (used in production)
  adapter_config.json            # training hyperparameters
```

### 6.2  How adapters are applied at inference time

```
mlx_lm server --model <base_model> --adapter-path data/adapters/technical_v1/
```

The base weights are frozen; the adapter matrices $A$ and $B$ are loaded on top.
The effective weight at each adapted layer is $W_{\text{eff}} = W_{\text{base}} + \Delta W$
where $\Delta W = A B \cdot \frac{\alpha}{r}$ (scaled by the LoRA $\alpha$ parameter).

Because $\Delta W$ has only $r(d + k)$ free parameters (vs. $d \times k$ for full fine-tuning),
the adapter for a 32B model at rank 8 is **~230 MB** vs. ~32 GB for full fine-tuning.

In [ ]:
"""Inspect adapter config and file sizes."""
import os

for name in ["technical_v1", "fundamental_v1"]:
    adir = ADAPTERS / name
    if not adir.exists():
        print(f"  {name}/  NOT FOUND (run: make finetune-train)")
        continue

    files = sorted(adir.iterdir())
    total_mb = sum(f.stat().st_size for f in files if f.is_file()) / 1e6
    final_mb = (adir / "adapters.safetensors").stat().st_size / 1e6 \
               if (adir / "adapters.safetensors").exists() else 0

    print(f"\n{name}/  ({total_mb:.1f} MB total, {len(files)} files)")
    for f in files:
        print(f"  {f.name:<42} {f.stat().st_size/1e6:>6.1f} MB")

    cfg_path = adir / "adapter_config.json"
    if cfg_path.exists():
        cfg = json.loads(cfg_path.read_text())
        lora = cfg.get("lora_parameters", {})
        eff_batch = cfg.get("batch_size", 1) * cfg.get("grad_accumulation_steps", 1)
        print(f"  --- config ---")
        print(f"  rank={lora.get('rank')}  alpha={lora.get('scale')}  "
              f"dropout={lora.get('dropout')}")
        print(f"  iters={cfg.get('iters')}  lr={cfg.get('learning_rate')}  "
              f"eff_batch={eff_batch}  max_seq={cfg.get('max_seq_length')}")
        print(f"  num_layers_adapted={cfg.get('num_layers')} / 64")
        print(f"  final adapter: {final_mb:.1f} MB  " +
              f"(full fine-tune would be ~{64*1024:,} MB)")

---
## 7  Three-Tier Evaluation (OQ-M02)

### 7.1  Framework (DJ-058)

Three independent dimensions are measured. A fine-tuned model is **not deployed**
unless Tier 1 and Tier 3 pass.

| Tier | Dimension | Key metric | Gate? | Pass criterion |
|---|---|---|---|---|
| 1 | Individual quality | HR, GR (Phase 5 verifier) | Yes | HR must not increase; Technical GR ≥ 0.717 |
| 2 | Collective accuracy | Method accuracy (Phase 10 labeler) | No | Improvement desirable |
| 3 | Ensemble diversity | pairwise_diversity | Yes | ≥ 0.9 × base |

**Why Tier 3 gates deployment:** If fine-tuning raises $\rho$ so much that
$E_{\text{ens}} > E_{\text{single}}$, we have traded a diversity advantage for
marginal individual improvement — a net negative for the collective.

### 7.2  Run protocol

```bash
# Start all servers
# LM Studio: open GUI, load qwen2.5-coder-32b-instruct-mlx on port 1234
make finetune-serve       # starts ports 1235 (technical) and 1236 (fundamental)

# Run evaluation  (AAPL, JPM, XOM at 2023-03-31)
uv run python scripts/run_phase11_evaluation.py

# Stop fine-tuned servers
make finetune-stop
```

Or as a single command: `make baseline-phase11`  
Output: `tests/fixtures/baseline/phase11_evaluation.json`

In [ ]:
"""Load Phase 11 evaluation fixture and visualise three-tier results."""
p11_path = FIXTURES / "phase11_evaluation.json"

if not p11_path.exists():
    print(f"Evaluation fixture not found: {p11_path}")
    print("Run:  make baseline-phase11")
    print()
    print("Phase 5 baseline (reference, base model):")
    print("  Technical GR (mean across AAPL/JPM/XOM): 0.667")
    print("  Fundamental GR (mean):                   1.000")
    print("  Target for Phase 11:                     Technical GR >= 0.717")
else:
    p11 = json.loads(p11_path.read_text())
    results = p11.get("results", [])
    meta    = p11.get("metadata", {})

    print(f"Evaluation date:   {meta.get('analysis_date')}")
    print(f"Tickers:           {meta.get('tickers')}")
    print(f"Run at:            {meta.get('run_date', 'unknown')}")
    print()

    tickers_ev = [r["ticker"] for r in results]
    base_tg    = [r["base_technical_gr"]      for r in results]
    ft_tg      = [r.get("finetuned_technical_gr", 0) or 0 for r in results]
    base_fg    = [r["base_fundamental_gr"]    for r in results]
    ft_fg      = [r.get("finetuned_fundamental_gr", 0) or 0 for r in results]
    base_div   = [r.get("base_pairwise_diversity", 0) or 0 for r in results]
    ft_div     = [r.get("finetuned_pairwise_diversity", 0) or 0 for r in results]
    div_thresh = [0.9 * b for b in base_div]

    # Pretty table
    print(f"{'Ticker':>8} {'BaseT-GR':>10} {'FT-GR':>8} {'D-GR':>8} "
          f"{'Base-Div':>10} {'FT-Div':>8} {'D-OK':>6}")
    print("-" * 64)
    for r in results:
        diff_gr  = (r.get("finetuned_technical_gr") or 0) - r["base_technical_gr"]
        diff_div = (r.get("finetuned_pairwise_diversity") or 0)
        ok = "YES" if r.get("diversity_preserved") else "NO"
        print(f"{r['ticker']:>8} {r['base_technical_gr']:>10.3f} "
              f"{r.get('finetuned_technical_gr') or 0:>8.3f} "
              f"{diff_gr:>+8.3f} "
              f"{r.get('base_pairwise_diversity') or 0:>10.3f} "
              f"{r.get('finetuned_pairwise_diversity') or 0:>8.3f} "
              f"{ok:>6}")

    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    x = np.arange(len(tickers_ev))
    w = 0.35

    # Tier 1: Technical GR
    ax = axes[0]
    ax.bar(x - w/2, base_tg, w, color="steelblue", alpha=0.82, label="Base (Phase 5)")
    ax.bar(x + w/2, ft_tg,  w, color="seagreen",   alpha=0.82, label="Fine-tuned")
    ax.axhline(0.717, color="orange", lw=1.4, ls="--", label="Target GR=0.717")
    ax.set(xticks=x, xticklabels=tickers_ev, ylabel="Grounding Rate (GR)",
           ylim=(0, 1.1), title="Tier 1: Technical Agent GR")
    ax.legend(fontsize=8)

    # Tier 3: Diversity
    ax2 = axes[1]
    ax2.bar(x - w/2, base_div, w, color="steelblue", alpha=0.82, label="Base")
    ax2.bar(x + w/2, ft_div,  w, color="seagreen",  alpha=0.82, label="Fine-tuned")
    for i, thresh in enumerate(div_thresh):
        ax2.plot([i - w, i + w], [thresh, thresh], color="orange", lw=1.4, ls="--")
    ax2.set(xticks=x, xticklabels=tickers_ev, ylabel="Pairwise diversity",
            ylim=(0, 1.1), title="Tier 3: Diversity (orange = 0.9 x base threshold)")
    ax2.legend(fontsize=8)

    # Heatmap summary
    ax3 = axes[2]
    tier_ok = [
        [1.0 if (r.get("finetuned_technical_gr") or 0) >= 0.717 else 0.2 for r in results],
        [0.6] * len(results),  # Tier 2 partial (not fully evaluated offline)
        [1.0 if r.get("diversity_preserved") else 0.2 for r in results],
    ]
    im = ax3.imshow(tier_ok, cmap="RdYlGn", vmin=0, vmax=1, aspect="auto")
    ax3.set(xticks=range(len(tickers_ev)), xticklabels=tickers_ev,
            yticks=[0, 1, 2],
            yticklabels=["Tier 1\nGR", "Tier 2\nAccuracy", "Tier 3\nDiversity"],
            title="Pass/fail summary")
    for i in range(3):
        for j in range(len(tickers_ev)):
            v = tier_ok[i][j]
            txt = "PASS" if v >= 0.8 else ("PARTIAL" if v >= 0.5 else "FAIL")
            ax3.text(j, i, txt, ha="center", va="center",
                     fontsize=8, color="white" if v < 0.5 or v >= 0.8 else "black")

    plt.suptitle("Figure 7: Phase 11 three-tier evaluation results", fontsize=11)
    plt.tight_layout()
    plt.show()

    all_tech_ok = all(r.get("gr_improved_technical", False) for r in results)
    all_div_ok  = all(r.get("diversity_preserved",    False) for r in results)
    print(f"\nOQ-M01 — Technical GR improved:   {'YES' if all_tech_ok else 'NO'}")
    print(f"OQ-M02 — Diversity preserved:      {'YES' if all_div_ok  else 'NO'}")
    print(f"Deploy fine-tuned models:          "
          + ("YES (both gates pass)" if all_tech_ok and all_div_ok
             else "NO (at least one gate failed)"))

---
## 8  Replication Guide and Open Questions

### 8.1  Step-by-step replication

All commands run from the repo root. Internet required only for Step 1.

```bash
# Step 1: Data (internet, ~5 min)
make acquire-data-phase10           # 15 tickers, 2016-2023, ~20 MB Parquet

# Step 2: Reference labels (CPU only, ~2 min)
make generate-reference-strategies  # data/reference_strategies/**/*.parquet

# Step 3: Training JSONL (CPU only, ~1 min)
uv run python scripts/generate_training_jsonl.py
uv run python scripts/generate_compliance_examples.py

# Step 4: Fine-tuning virtualenv (Apple Silicon only)
make finetune-setup                 # venvs/finetune/ with mlx==0.31.1, mlx_lm==0.31.1

# Step 5: Rank sweep (~165 min on M3 Ultra)
uv run python scripts/run_phase11_finetune.py --rank-sweep --sweep-iters 300
uv run python scripts/analyze_rank_sweep.py     # prints table + recommendation

# Step 6: Full training (~3.5 h on M3 Ultra)
uv run python scripts/run_phase11_finetune.py   # uses optimal rank from sweep

# Step 7: Three-tier evaluation (requires LM Studio GUI)
make baseline-phase11               # starts servers, runs eval, stops servers
```

### 8.2  Hardware requirements

| Component | Minimum | Tested |
|---|---|---|
| Chip | Apple Silicon M1 Ultra | M3 Ultra |
| Unified memory | 64 GB | 192 GB |
| Peak VRAM (training) | ~52 GB | 52 GB |
| Peak VRAM (3 servers) | ~156 GB | 156 GB |

At 64 GB, only one fine-tuned server can run alongside LM Studio.
Evaluate Tier 1 (technical) and Tier 1 (fundamental) in two separate runs.

### 8.3  Key findings (summary)

| Finding | Value |
|---|---|
| Optimal LoRA rank | 8 |
| Technical training loss (1000 iters, rank 8) | 0.299 |
| Fundamental training loss (1000 iters, rank 8) | 0.299 |
| Technical adapter size | ~230 MB |
| Fundamental adapter size | ~230 MB |
| Training time (technical) | 8202 s (~2.3 h) |
| Training time (fundamental) | 2767 s (~46 min) |
| OQ-M01 (rank selection) | Rank 8 confirmed |
| OQ-M02 (diversity) | See Section 7 results |

### 8.4  Open questions for Phase 12

1. **GraphRAG complement:** The fine-tuned fundamental agent was trained *without*
   RAG context. Phase 12 will measure whether GraphRAG adds value on top of domain
   fine-tuning, or whether one subsumes the other.

2. **Structured debate after fine-tuning:** Phase 11 agents have different decision
   boundaries. Does adversarial deliberation between them improve accuracy, or cause
   polarisation? Phase 12 designs the debate mechanism.

3. **Bear market validation:** 2016-2022 labels are 61% Buy. Including 2022-Q4
   (bear market) would test whether the adapter generalises to Sell regimes.

4. **DPO alignment:** Phase 5 verification reports distinguish HR=0 (perfect) from
   HR>0 (hallucinated) outputs. These form natural preference pairs for DPO training
   — a future alternative to supervised fine-tuning.

In [ ]:
"""Print a final phase summary — confirms all artifacts are present."""
checks = [
    (REF_STRAT / "max_return" / "AAPL_60d.parquet",          "Reference labels (AAPL)"),
    (TRAINING / "technical_max_return_60d.jsonl",             "Technical JSONL"),
    (TRAINING / "fundamental_risk_adjusted_60d.jsonl",        "Fundamental JSONL"),
    (TRAINING / "rank_sweep_results.json",                    "Rank sweep results"),
    (TRAINING / "optimal_rank.json",                          "Optimal rank"),
    (ADAPTERS / "technical_v1" / "adapters.safetensors",      "technical_v1 adapter"),
    (ADAPTERS / "fundamental_v1" / "adapters.safetensors",    "fundamental_v1 adapter"),
    (FIXTURES / "phase11_evaluation.json",                    "Three-tier evaluation fixture"),
]

print("Phase 11 artifact checklist")
print("=" * 52)
all_ok = True
for path, label in checks:
    ok = path.exists()
    status = "OK" if ok else "MISSING"
    print(f"  {'[OK]' if ok else '[ ]':5}  {label}")
    if not ok:
        all_ok = False

print()
if all_ok:
    print("All artifacts present. Phase 11 is fully replicated.")
else:
    print("Some artifacts missing. See Section 8.1 for the make targets to generate them.")